# LeetCode 2928 — Distribute Candies Among Children I bằng Python

Link đề: https://leetcode.com/problems/distribute-candies-among-children-i/ · Độ khó: **Easy** · Chủ đề: Math, Combinatorics, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def distributeCandies(self, n: int, limit: int) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chia `n` viên kẹo cho **3 đứa trẻ** sao cho **không đứa nào nhận quá `limit` viên**; đếm **số cách chia** (các đứa trẻ phân biệt nhau, nên `(1, 2, 2)` và `(2, 1, 2)` là hai cách khác nhau).
2. Biết **liệt kê có ràng buộc** bằng vòng lặp lồng nhau, và cách **giảm số vòng lặp** khi một biến được xác định bởi các biến còn lại (`c = n - a - b`).
3. Đi từ cách **ba vòng lặp `O(limit³)`** → **hai vòng lặp và suy ra `c`** → **một vòng lặp, đếm số cặp `(b, c)` bằng công thức** → **công thức tổ hợp bao hàm - loại trừ `O(1)`** → **quy hoạch động theo số đứa trẻ**.
4. Hiểu **kỹ thuật "chia kẹo" (stars and bars)** và **nguyên lý bao hàm - loại trừ**: đếm tất cả cách chia, trừ đi các cách có đứa nhận quá `limit`.
5. Tránh các lỗi kinh điển: dùng `range(limit)` (thiếu giá trị `limit`), quên điều kiện `c >= 0`, nhầm sang đếm **tổ hợp không thứ tự**, quên cộng lại phần giao khi loại trừ, và dùng `comb` với số âm.

## 2. Đề bài & ví dụ

Cho hai số nguyên dương `n` và `limit`. Hãy trả về **tổng số cách** chia `n` viên kẹo cho **3 đứa trẻ** sao cho **không đứa trẻ nào nhận nhiều hơn `limit` viên** (mỗi đứa nhận số viên **nguyên không âm**, và tổng đúng bằng `n`). Ba đứa trẻ **phân biệt** nhau (đứa thứ nhất, thứ hai, thứ ba), nên **thứ tự quan trọng**.

**Ví dụ 1**

- Input: `n = 5`, `limit = 2`
- Output: `3`
- Giải thích: ba cách chia là `(1, 2, 2)`, `(2, 1, 2)`, `(2, 2, 1)` (mỗi đứa tối đa 2 viên, tổng 5).

**Ví dụ 2**

- Input: `n = 3`, `limit = 3`
- Output: `10`
- Giải thích: có 10 cách chia 3 viên cho 3 đứa mà không đứa nào quá 3 viên: `(0,0,3)`, `(0,1,2)`, `(0,2,1)`, `(0,3,0)`, `(1,0,2)`, `(1,1,1)`, `(1,2,0)`, `(2,0,1)`, `(2,1,0)`, `(3,0,0)`.

**Ràng buộc:** `1 <= n <= 50`; `1 <= limit <= 50`.

In [1]:
from typing import List
from math import comb
import random

examples = [
    (5, 2, 3),
    (3, 3, 10),
]
for n, limit, expected in examples:
    print(f"n={n}, limit={limit} -> kỳ vọng {expected}")

n=5, limit=2 -> kỳ vọng 3
n=3, limit=3 -> kỳ vọng 10


## 3. Kiến thức nền

### 3.1. Bộ ba `(a, b, c)` là một cách chia

Gọi `a, b, c` là số kẹo của ba đứa trẻ. Một cách chia hợp lệ là bộ ba **số nguyên** `(a, b, c)` thỏa:

- `0 <= a, b, c <= limit`;
- `a + b + c == n`.

Hai bộ ba khác nhau ở **bất kỳ vị trí nào** là hai cách khác nhau (các đứa trẻ phân biệt), nên `(1, 2, 2)` khác `(2, 1, 2)`.

In [2]:
n, limit = 5, 2
def valid(a, b, c):
    return 0 <= a <= limit and 0 <= b <= limit and 0 <= c <= limit and a + b + c == n

print(valid(1, 2, 2), valid(2, 2, 1))    # True True
print(valid(3, 1, 1), valid(0, 2, 2))    # False (3 > limit)   False (tổng 4 != 5)

True True
False False


### 3.2. `range` và cận bao gồm: `range(limit + 1)`

`range(stop)` sinh `0, 1, ..., stop - 1` (**không gồm** `stop`). Mỗi đứa trẻ nhận được **từ `0` đến `limit` viên**, gồm cả `limit`, nên phải viết `range(limit + 1)`. `range(limit)` bỏ sót giá trị `limit`.

In [3]:
limit = 3
print(list(range(limit)))          # [0, 1, 2]: thiếu 3
print(list(range(limit + 1)))      # [0, 1, 2, 3]: đủ

[0, 1, 2]
[0, 1, 2, 3]


### 3.3. Ba vòng lặp lồng nhau và đếm

Ta duyệt mọi `(a, b, c)` với mỗi biến chạy `0..limit` và **đếm** các bộ ba có tổng `n`. Số bộ ba cần duyệt là `(limit + 1)³` (với `limit = 50` là `132 651`).

In [4]:
n, limit = 5, 2
count = 0
solutions = []
for a in range(limit + 1):
    for b in range(limit + 1):
        for c in range(limit + 1):
            if a + b + c == n:
                count += 1
                solutions.append((a, b, c))
print(count, solutions)

3 [(1, 2, 2), (2, 1, 2), (2, 2, 1)]


### 3.4. Suy ra biến thứ ba: `c = n - a - b`

Khi đã chọn `a` và `b`, số kẹo của đứa thứ ba **bị xác định**: `c = n - a - b`. Ta chỉ cần kiểm tra `c` có **hợp lệ** không: `0 <= c <= limit`. Nhờ vậy bỏ được một vòng lặp.

In [5]:
n, limit = 5, 2
for a in range(limit + 1):
    for b in range(limit + 1):
        c = n - a - b
        ok = 0 <= c <= limit
        if ok:
            print(f"a={a}, b={b} -> c={c} hợp lệ")

a=1, b=2 -> c=2 hợp lệ
a=2, b=1 -> c=2 hợp lệ
a=2, b=2 -> c=1 hợp lệ


### 3.5. Đếm cặp `(b, c)` với `b + c = r` và `0 <= b, c <= limit`

Khi `a` đã cố định, còn `r = n - a` kẹo chia cho hai đứa còn lại. Số cặp `(b, c)` với `b + c = r`, `0 <= b, c <= limit`:

- `b` phải thỏa `0 <= b <= limit` và `c = r - b` thỏa `0 <= r - b <= limit`, tức `r - limit <= b <= r`;
- nên `b` chạy từ `max(0, r - limit)` đến `min(r, limit)`;
- số giá trị `b` là `max(0, min(r, limit) - max(0, r - limit) + 1)`.

Công thức cho số cách trong **`O(1)`** thay vì duyệt `b`.

In [6]:
def count_pairs(r, limit):
    lo = max(0, r - limit)
    hi = min(r, limit)
    return max(0, hi - lo + 1)


for r in range(0, 7):
    brute = sum(1 for b in range(limit + 1) for c in range(limit + 1) if b + c == r)
    print(f"r={r}: công thức={count_pairs(r, limit)} đếm tay={brute}")
    assert count_pairs(r, limit) == brute

r=0: công thức=1 đếm tay=1
r=1: công thức=2 đếm tay=2
r=2: công thức=3 đếm tay=3
r=3: công thức=2 đếm tay=2
r=4: công thức=1 đếm tay=1
r=5: công thức=0 đếm tay=0
r=6: công thức=0 đếm tay=0


### 3.6. "Chia kẹo" không giới hạn: stars and bars

Số cách chia `m` viên kẹo cho **3 đứa trẻ phân biệt**, mỗi đứa nhận số nguyên **không âm** (không có `limit`) là **`C(m + 2, 2)`**. Lý do: xếp `m` ngôi sao `*` và `2` vách ngăn `|` trên một hàng: ví dụ `**|*|*` là `(2, 1, 1)`; số cách chọn vị trí hai vách ngăn trong `m + 2` vị trí là `C(m + 2, 2)`. Với `m < 0` thì số cách là `0`.

In [7]:
def free_ways(m):
    return comb(m + 2, 2) if m >= 0 else 0        # số cách chia m kẹo cho 3 đứa, không giới hạn


for m in [0, 1, 2, 3, 5]:
    brute = sum(1 for a in range(m + 1) for b in range(m + 1) if a + b <= m)    # c = m - a - b >= 0
    print(f"m={m}: C(m+2,2)={free_ways(m)} đếm tay={brute}")
    assert free_ways(m) == brute
print(free_ways(-1))      # 0

m=0: C(m+2,2)=1 đếm tay=1
m=1: C(m+2,2)=3 đếm tay=3
m=2: C(m+2,2)=6 đếm tay=6
m=3: C(m+2,2)=10 đếm tay=10
m=5: C(m+2,2)=21 đếm tay=21
0


### 3.7. Nguyên lý bao hàm - loại trừ

Ta muốn đếm cách chia mà **cả ba đứa đều `<= limit`**. Gọi `A_i` là tập cách chia mà **đứa `i` nhận `>= limit + 1`** viên ("vi phạm"). Số cách hợp lệ = tất cả − các cách có ít nhất một vi phạm:

`ans = |U| − |A₁ ∪ A₂ ∪ A₃|`

và theo bao hàm - loại trừ:

`|A₁ ∪ A₂ ∪ A₃| = Σ|Aᵢ| − Σ|Aᵢ ∩ Aⱼ| + |A₁ ∩ A₂ ∩ A₃|`.

Mỗi tập giao đếm bằng stars and bars: nếu `j` đứa **được chỉ định vi phạm** (mỗi đứa ít nhất `limit + 1` viên), ta **bớt trước** `j·(limit + 1)` viên đó rồi chia phần còn lại tự do: `C(n − j(limit+1) + 2, 2)`.

In [8]:
n, limit = 5, 2
total = free_ways(n)                                   # tất cả: 21
one_bad = 3 * free_ways(n - (limit + 1))               # 3 cách chọn đứa vi phạm
two_bad = 3 * free_ways(n - 2 * (limit + 1))           # 3 cặp đứa vi phạm
three_bad = 1 * free_ways(n - 3 * (limit + 1))         # cả 3 đứa vi phạm
print(total, one_bad, two_bad, three_bad)              # 21 18 0 0
print("hợp lệ =", total - one_bad + two_bad - three_bad)   # 3

21 18 0 0
hợp lệ = 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `n = 5`, `limit = 2`, ta liệt kê theo đứa thứ nhất `a` rồi đếm cách chia `r = 5 - a` kẹo cho hai đứa còn lại (mỗi đứa tối đa 2):

| `a` | `r = 5 - a` | các cặp `(b, c)` với `b + c = r`, `b, c <= 2` | số cách |
|---|---|---|---|
| 0 | 5 | không có (tối đa 2 + 2 = 4 < 5) | 0 |
| 1 | 4 | `(2, 2)` | 1 |
| 2 | 3 | `(1, 2)`, `(2, 1)` | 2 |

(Với `a > 2` không hợp lệ vì vượt `limit`.) Tổng `0 + 1 + 2 = 3`. Thuật toán: **duyệt số kẹo `a` của đứa đầu tiên, đếm số cách chia phần còn lại cho hai đứa kia**.

## 5. Cách 1 — Brute Force / baseline: ba vòng lặp

**Ý tưởng.** Duyệt mọi `(a, b, c)` với `0 <= a, b, c <= limit` và đếm các bộ có `a + b + c == n`. Mô phỏng đúng định nghĩa của đề.

In [9]:
def distribute_triple_loop(n: int, limit: int) -> int:
    count = 0
    for a in range(limit + 1):
        for b in range(limit + 1):
            for c in range(limit + 1):
                if a + b + c == n:
                    count += 1
    return count


for n, limit, expected in examples:
    print(n, limit, "->", distribute_triple_loop(n, limit), "(kỳ vọng", expected, ")")

5 2 -> 3 (kỳ vọng 3 )
3 3 -> 10 (kỳ vọng 10 )


In [10]:
# Liệt kê các cách chia của ví dụ 2 để đối chiếu với đề
n, limit = 3, 3
ways = [(a, b, c) for a in range(limit + 1) for b in range(limit + 1) for c in range(limit + 1) if a + b + c == n]
print(len(ways), ways)

10 [(0, 0, 3), (0, 1, 2), (0, 2, 1), (0, 3, 0), (1, 0, 2), (1, 1, 1), (1, 2, 0), (2, 0, 1), (2, 1, 0), (3, 0, 0)]


**Độ phức tạp (Cách 1).** Ba vòng lặp mỗi vòng `limit + 1` lần: **Time `O(limit³)`** (với `limit = 50`: khoảng `132 651` phép); chỉ một biến đếm: **Space `O(1)`**.

**Nút thắt.** Ta duyệt cả những bộ `(a, b, c)` có tổng sai. Biến `c` hoàn toàn **bị quyết định** bởi `a, b` và `n`.

## 6. Ý tưởng thuật toán / bước đột phá

**Giảm số vòng lặp.** Khi `a` và `b` đã chọn thì `c = n - a - b`, nên chỉ cần kiểm tra `0 <= c <= limit` (Cách 2). Tiếp tục: khi `a` đã chọn, **số cặp `(b, c)`** có thể đếm bằng công thức `O(1)` (Cách 3). Cuối cùng, nhìn tổng thể bằng **tổ hợp** và bao hàm - loại trừ, ta có công thức đóng **không vòng lặp** (Cách 4).

Cùng một bài, bốn mức "hiểu": liệt kê, suy ra biến, đếm theo công thức từng phần, và công thức đóng. Phiên bản **II** của bài (cùng series) có `n, limit` tới `10^6`, nên cần Cách 3 hoặc Cách 4.

## 7. Các cách tiếp theo

### Cách 2 — Hai vòng lặp, suy ra `c = n - a - b`

**Giải thích.** Duyệt `a, b` và tính `c = n - a - b`; đếm nếu `0 <= c <= limit`.

In [11]:
def distribute_double_loop(n: int, limit: int) -> int:
    count = 0
    for a in range(limit + 1):
        for b in range(limit + 1):
            c = n - a - b                   # đứa thứ ba bị quyết định
            if 0 <= c <= limit:
                count += 1
    return count


for n, limit, expected in examples:
    print(n, limit, "->", distribute_double_loop(n, limit), "(kỳ vọng", expected, ")")

5 2 -> 3 (kỳ vọng 3 )
3 3 -> 10 (kỳ vọng 10 )


In [12]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def distributeCandies(self, n: int, limit: int) -> int:
        count = 0
        for a in range(limit + 1):
            for b in range(limit + 1):
                c = n - a - b
                if 0 <= c <= limit:
                    count += 1
        return count

**Độ phức tạp Cách 2:** **Time `O(limit²)`**, **Space `O(1)`**.

### Cách 3 — Một vòng lặp, đếm số cặp `(b, c)` bằng công thức

**Giải thích.** Duyệt `a` từ `0` đến `min(n, limit)`; với `r = n - a`, số cặp `(b, c)` là `max(0, min(r, limit) - max(0, r - limit) + 1)`. Cộng dồn.

In [13]:
def distribute_single_loop(n: int, limit: int) -> int:
    count = 0
    for a in range(min(n, limit) + 1):          # a không thể vượt n hoặc limit
        r = n - a                                # kẹo còn lại cho hai đứa còn lại
        lo = max(0, r - limit)                   # b nhỏ nhất sao cho c = r - b <= limit
        hi = min(r, limit)                       # b lớn nhất sao cho c = r - b >= 0
        count += max(0, hi - lo + 1)
    return count


for n, limit, expected in examples:
    print(n, limit, "->", distribute_single_loop(n, limit), "(kỳ vọng", expected, ")")

5 2 -> 3 (kỳ vọng 3 )
3 3 -> 10 (kỳ vọng 10 )


In [14]:
# Code nộp LeetCode cho Cách 3
class Solution:
    def distributeCandies(self, n: int, limit: int) -> int:
        count = 0
        for a in range(min(n, limit) + 1):
            r = n - a
            count += max(0, min(r, limit) - max(0, r - limit) + 1)
        return count

**Độ phức tạp Cách 3:** **Time `O(limit)`** (hoặc `O(min(n, limit))`), **Space `O(1)`**. Đủ nhanh cho Bài II (`10^6`).

### Cách 4 — Công thức đóng bằng bao hàm - loại trừ `O(1)`

**Giải thích.** `ans = Σ_{j=0..3} (-1)^j · C(3, j) · F(n − j(limit+1))` với `F(m) = C(m + 2, 2)` nếu `m >= 0`, ngược lại `0`. Dấu xen kẽ `+ − + −`, hệ số `C(3, j) = 1, 3, 3, 1`.

In [15]:
def distribute_inclusion_exclusion(n: int, limit: int) -> int:
    def free(m):                               # chia m kẹo cho 3 đứa, không giới hạn
        return comb(m + 2, 2) if m >= 0 else 0

    return (free(n)
            - 3 * free(n - (limit + 1))
            + 3 * free(n - 2 * (limit + 1))
            - free(n - 3 * (limit + 1)))


for n, limit, expected in examples:
    print(n, limit, "->", distribute_inclusion_exclusion(n, limit), "(kỳ vọng", expected, ")")

5 2 -> 3 (kỳ vọng 3 )
3 3 -> 10 (kỳ vọng 10 )


In [16]:
# Code nộp LeetCode cho Cách 4
from math import comb

class Solution:
    def distributeCandies(self, n: int, limit: int) -> int:
        def free(m: int) -> int:
            return comb(m + 2, 2) if m >= 0 else 0
        return free(n) - 3 * free(n - (limit + 1)) + 3 * free(n - 2 * (limit + 1)) - free(n - 3 * (limit + 1))

**Độ phức tạp Cách 4:** **Time `O(1)`** (4 lần gọi `comb` trên số nhỏ), **Space `O(1)`**.

### Cách 5 — Quy hoạch động theo số đứa trẻ

**Giải thích.** `dp[k][s]` = số cách chia `s` viên cho `k` đứa đầu tiên (mỗi đứa `<= limit`). Khởi tạo `dp[0][0] = 1`. Chuyển: `dp[k][s] = Σ_{x=0..min(limit, s)} dp[k-1][s-x]`. Đáp án `dp[3][n]`. Cách này **tổng quát cho số đứa trẻ bất kỳ**, nên đáng học dù không nhanh nhất.

In [17]:
def distribute_dp(n: int, limit: int, children: int = 3) -> int:
    dp = [0] * (n + 1)                           # dp[s] sau k đứa; ban đầu k = 0
    dp[0] = 1
    for _ in range(children):
        new = [0] * (n + 1)
        for s in range(n + 1):
            for x in range(min(limit, s) + 1):   # đứa mới nhận x viên
                new[s] += dp[s - x]
        dp = new
    return dp[n]


for n, limit, expected in examples:
    print(n, limit, "->", distribute_dp(n, limit), "(kỳ vọng", expected, ")")
print("4 đứa trẻ, n=5, limit=2:", distribute_dp(5, 2, children=4))

5 2 -> 3 (kỳ vọng 3 )
3 3 -> 10 (kỳ vọng 10 )
4 đứa trẻ, n=5, limit=2: 16


**Độ phức tạp Cách 5:** **Time `O(children · n · limit)`**, **Space `O(n)`**.

> Cách 2 (hai vòng lặp) là cách nộp dễ viết nhất; Cách 3 và 4 mới là cách mở rộng được cho `n` rất lớn (Bài II).

## 8. TRACE từng bước

Ta theo dõi (a) Cách 3: từng `a` và số cặp `(b, c)`; (b) Cách 4: từng số hạng của bao hàm - loại trừ.

In [18]:
def trace_single_loop(n, limit):
    total = 0
    print(f"n={n}, limit={limit}")
    for a in range(min(n, limit) + 1):
        r = n - a
        lo = max(0, r - limit)
        hi = min(r, limit)
        pairs = max(0, hi - lo + 1)
        total += pairs
        pair_list = [(b, r - b) for b in range(lo, hi + 1)] if pairs else []
        print(f"a={a}: r={r}, b trong [{lo}, {hi}] -> {pairs} cặp {pair_list} | total={total}")
    print("=>", total)
    return total


assert trace_single_loop(5, 2) == 3

n=5, limit=2
a=0: r=5, b trong [3, 2] -> 0 cặp [] | total=0
a=1: r=4, b trong [2, 2] -> 1 cặp [(2, 2)] | total=1
a=2: r=3, b trong [1, 2] -> 2 cặp [(1, 2), (2, 1)] | total=3
=> 3


In [19]:
def trace_inclusion_exclusion(n, limit):
    def free(m):
        return comb(m + 2, 2) if m >= 0 else 0
    print(f"n={n}, limit={limit}, limit+1={limit + 1}")
    total = 0
    for j, coef in enumerate([1, 3, 3, 1]):
        m = n - j * (limit + 1)
        term = coef * free(m)
        sign = 1 if j % 2 == 0 else -1
        total += sign * term
        print(f"j={j}: hệ số C(3,{j})={coef}, m=n-j(limit+1)={m}, F(m)={free(m)} -> {'+' if sign > 0 else '-'} {term} | tổng tạm = {total}")
    print("=>", total)
    return total


assert trace_inclusion_exclusion(5, 2) == 3
print()
assert trace_inclusion_exclusion(3, 3) == 10

n=5, limit=2, limit+1=3
j=0: hệ số C(3,0)=1, m=n-j(limit+1)=5, F(m)=21 -> + 21 | tổng tạm = 21
j=1: hệ số C(3,1)=3, m=n-j(limit+1)=2, F(m)=6 -> - 18 | tổng tạm = 3
j=2: hệ số C(3,2)=3, m=n-j(limit+1)=-1, F(m)=0 -> + 0 | tổng tạm = 3
j=3: hệ số C(3,3)=1, m=n-j(limit+1)=-4, F(m)=0 -> - 0 | tổng tạm = 3
=> 3

n=3, limit=3, limit+1=4
j=0: hệ số C(3,0)=1, m=n-j(limit+1)=3, F(m)=10 -> + 10 | tổng tạm = 10
j=1: hệ số C(3,1)=3, m=n-j(limit+1)=-1, F(m)=0 -> - 0 | tổng tạm = 10
j=2: hệ số C(3,2)=3, m=n-j(limit+1)=-5, F(m)=0 -> + 0 | tổng tạm = 10
j=3: hệ số C(3,3)=1, m=n-j(limit+1)=-9, F(m)=0 -> - 0 | tổng tạm = 10
=> 10


Bảng tóm tắt (Cách 4, `n = 5`, `limit = 2`):

| `j` | dấu | `C(3, j)` | `m = n − j·3` | `F(m) = C(m+2, 2)` | số hạng |
|---|---|---|---|---|---|
| 0 | + | 1 | 5 | 21 | +21 |
| 1 | − | 3 | 2 | 6 | −18 |
| 2 | + | 3 | −1 | 0 | 0 |
| 3 | − | 1 | −4 | 0 | 0 |

Tổng `21 − 18 + 0 − 0 = 3`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (một vòng lặp, đếm cặp bằng công thức): nhanh, ít rủi ro sai dấu hơn bao hàm - loại trừ. Đúng class `Solution`, đúng method `distributeCandies(self, n: int, limit: int) -> int`.

In [20]:
class Solution:
    def distributeCandies(self, n: int, limit: int) -> int:
        count = 0
        for a in range(min(n, limit) + 1):          # kẹo của đứa thứ nhất
            r = n - a                                # kẹo còn lại cho hai đứa kia
            lo = max(0, r - limit)                   # b nhỏ nhất
            hi = min(r, limit)                       # b lớn nhất
            count += max(0, hi - lo + 1)
        return count


sol = Solution()
for n, limit, expected in examples:
    print(n, limit, "->", sol.distributeCandies(n, limit))

5 2 -> 3
3 3 -> 10


## 10. Vì sao thuật toán đúng

- **Cách 1, 2 (liệt kê):** duyệt mọi `(a, b)` (và `c`) hợp lệ và đếm đúng các bộ thỏa `a + b + c = n`, `0 <= a, b, c <= limit`; mỗi cách chia là một bộ ba duy nhất nên đếm không trùng, không sót.
- **Cách 3:** phân hoạch các cách chia theo giá trị `a` của đứa thứ nhất. Với `a` cố định, `(b, c)` thỏa `b + c = r`, `0 <= b, c <= limit`; `b` nhận mọi giá trị nguyên trong `[max(0, r − limit), min(r, limit)]` và `c = r − b` tự nằm trong `[0, limit]`. Số giá trị là `hi − lo + 1` (hoặc `0` nếu khoảng rỗng).
- **Cách 4:** bao hàm - loại trừ: `ans = |U| − Σ|Aᵢ| + Σ|Aᵢ ∩ Aⱼ| − |A₁ ∩ A₂ ∩ A₃|`. Với tập `S` gồm `j` đứa bị buộc nhận `>= limit + 1`, đặt `xᵢ' = xᵢ − (limit + 1)` cho mỗi đứa trong `S` ta chuyển về bài chia `n − j(limit + 1)` kẹo tự do, đếm bằng `C(·+2, 2)` (và `0` nếu âm).
- **Giá trị `a` tối đa:** `a <= min(n, limit)`; với `a` lớn hơn thì hoặc `r < 0` hoặc vượt `limit`.
- **Tính chất kiểm được:** đáp án bằng nhau cho cả bốn cách; đáp án `0` khi `n > 3·limit` (không đủ chỗ chứa); đáp án bằng `C(n + 2, 2)` khi `limit >= n` (không giới hạn thật sự).

## 11. Phân tích Time / Space Complexity

- **Cách 1:** Time `O(limit³)`, Space `O(1)`.
- **Cách 2:** Time `O(limit²)`, Space `O(1)`.
- **Cách 3:** Time `O(limit)`, Space `O(1)`.
- **Cách 4:** Time **`O(1)`**, Space `O(1)`.
- **Cách 5:** Time `O(children · n · limit)`, Space `O(n)`.
- Với ràng buộc `n, limit <= 50` mọi cách đều rất nhanh. Với Bài II (`10^6`), Cách 1 và 2 không còn đủ, Cách 3 (`O(limit)`) và Cách 4 (`O(1)`) vẫn chạy tốt.

## 12. Bộ test

Ta gom các cách vào từ điển `impls` rồi kiểm tra bằng `assert`.

In [21]:
sol = Solution()
impls = {
    "triple": distribute_triple_loop,
    "double": distribute_double_loop,
    "single": distribute_single_loop,
    "inclusion_exclusion": distribute_inclusion_exclusion,
    "dp": distribute_dp,
    "submit": sol.distributeCandies,
}

for n, limit, expected in examples:
    for key, fn in impls.items():
        assert fn(n, limit) == expected, (key, n, limit, fn(n, limit), expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [22]:
def check_all(n, limit, expected, label):
    for key, fn in impls.items():
        got = fn(n, limit)
        assert got == expected, (label, key, got, expected)
    print("PASS", label, f"(n={n}, limit={limit}) ->", expected)


check_all(1, 1, 3, "n = 1: chọn đứa nhận 1 viên")
check_all(1, 50, 3, "n = 1, limit lớn: vẫn 3 cách")
check_all(2, 1, 3, "n = 2, limit = 1: chọn đứa nhận 0 viên")
check_all(3, 1, 1, "n = 3, limit = 1: mỗi đứa đúng 1 viên")
check_all(4, 1, 0, "n > 3·limit: không có cách nào")
check_all(6, 2, 1, "n = 3·limit: mỗi đứa đúng limit")
check_all(7, 2, 0, "n > 3·limit")
check_all(50, 50, comb(52, 2), "n = limit = 50: không giới hạn thật sự, C(52, 2)")
check_all(50, 1, 0, "n lớn, limit nhỏ")

PASS n = 1: chọn đứa nhận 1 viên (n=1, limit=1) -> 3
PASS n = 1, limit lớn: vẫn 3 cách (n=1, limit=50) -> 3
PASS n = 2, limit = 1: chọn đứa nhận 0 viên (n=2, limit=1) -> 3
PASS n = 3, limit = 1: mỗi đứa đúng 1 viên (n=3, limit=1) -> 1
PASS n > 3·limit: không có cách nào (n=4, limit=1) -> 0
PASS n = 3·limit: mỗi đứa đúng limit (n=6, limit=2) -> 1
PASS n > 3·limit (n=7, limit=2) -> 0
PASS n = limit = 50: không giới hạn thật sự, C(52, 2) (n=50, limit=50) -> 1326
PASS n lớn, limit nhỏ (n=50, limit=1) -> 0


### 12.2. Tính chất và phản ví dụ

In [23]:
# Tính chất 1: limit >= n -> không giới hạn thật sự: C(n + 2, 2)
for n in range(1, 51):
    for limit in (n, n + 1, 50):
        if limit >= n:
            assert sol.distributeCandies(n, limit) == comb(n + 2, 2)
print("PASS limit >= n cho C(n + 2, 2)")

# Tính chất 2: n > 3 * limit -> 0; n == 3 * limit -> 1
for limit in range(1, 17):
    assert sol.distributeCandies(3 * limit + 1, limit) == 0
    assert sol.distributeCandies(3 * limit, limit) == 1
print("PASS n > 3·limit cho 0; n = 3·limit cho 1")

# Tính chất 3: đối xứng: số cách chia n kẹo = số cách chia 3·limit - n kẹo (bù: limit - x)
for limit in range(1, 20):
    for n in range(0, 3 * limit + 1):
        if 1 <= n <= 50 and 1 <= 3 * limit - n <= 50:
            assert sol.distributeCandies(n, limit) == sol.distributeCandies(3 * limit - n, limit)
print("PASS đối xứng n <-> 3·limit - n")

# Phản ví dụ 1: range(limit) thiếu giá trị limit
def sai_range_thieu(n, limit):
    return sum(1 for a in range(limit) for b in range(limit) for c in range(limit) if a + b + c == n)

assert sai_range_thieu(6, 2) == 0 and sol.distributeCandies(6, 2) == 1
print("PASS phản ví dụ: range(limit) cho", sai_range_thieu(6, 2), "thay vì 1")

# Phản ví dụ 2: đếm tổ hợp không thứ tự thay vì có thứ tự
def sai_khong_thu_tu(n, limit):
    seen = set()
    for a in range(limit + 1):
        for b in range(limit + 1):
            c = n - a - b
            if 0 <= c <= limit:
                seen.add(tuple(sorted((a, b, c))))
    return len(seen)

assert sai_khong_thu_tu(5, 2) == 1 != 3
print("PASS phản ví dụ: coi (1,2,2) và (2,1,2) là một cho", sai_khong_thu_tu(5, 2), "thay vì 3")

PASS limit >= n cho C(n + 2, 2)
PASS n > 3·limit cho 0; n = 3·limit cho 1
PASS đối xứng n <-> 3·limit - n
PASS phản ví dụ: range(limit) cho 0 thay vì 1
PASS phản ví dụ: coi (1,2,2) và (2,1,2) là một cho 1 thay vì 3


### 12.3. Đối chiếu toàn bộ miền của đề

In [24]:
# Mọi cặp (n, limit) với 1 <= n, limit <= 50 (2500 cặp)
for n in range(1, 51):
    for limit in range(1, 51):
        ref = distribute_triple_loop(n, limit)
        for key, fn in impls.items():
            assert fn(n, limit) == ref, (key, n, limit, fn(n, limit), ref)
print("PASS 2500 cặp (n, limit) trong miền của đề - mọi cách khớp nhau")

# Thử thêm giá trị lớn cho các cách nhanh (so Cách 3 với Cách 4)
random.seed(2928)
for _ in range(300):
    n, limit = random.randint(1, 5000), random.randint(1, 5000)
    assert distribute_single_loop(n, limit) == distribute_inclusion_exclusion(n, limit), (n, limit)
for n, limit in [(10**6, 10**6), (10**6, 400000), (10**6, 333334), (10**5, 50000)]:
    assert distribute_single_loop(n, limit) == distribute_inclusion_exclusion(n, limit), (n, limit)
print("PASS các cách O(limit) và O(1) khớp nhau cả với n, limit tới 10^6")

PASS 2500 cặp (n, limit) trong miền của đề - mọi cách khớp nhau


PASS các cách O(limit) và O(1) khớp nhau cả với n, limit tới 10^6


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `range(limit)` thiếu giá trị `limit`

In [25]:
def sai_range_thieu(n, limit):
    count = 0
    for a in range(limit):
        for b in range(limit):
            c = n - a - b
            if 0 <= c <= limit:
                count += 1
    return count


print("sai  :", sai_range_thieu(6, 2), "| đúng:", Solution().distributeCandies(6, 2))     # cách (2,2,2)

sai  : 0 | đúng: 1


**Vì sao sai:** mỗi đứa được nhận **tối đa `limit`** viên, gồm cả `limit`. `range(limit)` chỉ cho `0..limit-1`, bỏ sót cách chia `(2, 2, 2)` khi `limit = 2`. Dùng `range(limit + 1)`.

### Lỗi 2: Quên điều kiện `c >= 0`

In [26]:
def sai_quen_c_khong_am(n, limit):
    count = 0
    for a in range(limit + 1):
        for b in range(limit + 1):
            c = n - a - b
            if c <= limit:                          # thiếu c >= 0
                count += 1
    return count


print("sai  :", sai_quen_c_khong_am(3, 3), "| đúng:", Solution().distributeCandies(3, 3))

sai  : 16 | đúng: 10


**Vì sao sai:** nếu `c` âm (ví dụ `a = b = 3`, `n = 3` cho `c = -3`) thì đứa thứ ba "nhận" số kẹo âm, vô lý. Phải kiểm tra **cả hai cận**: `0 <= c <= limit`.

### Lỗi 3: Đếm tổ hợp không thứ tự

In [27]:
def sai_sort(n, limit):
    seen = set()
    for a in range(limit + 1):
        for b in range(limit + 1):
            c = n - a - b
            if 0 <= c <= limit:
                seen.add(tuple(sorted((a, b, c))))     # coi (1,2,2) = (2,1,2) = (2,2,1)
    return len(seen)


print("sai  :", sai_sort(5, 2), "| đúng:", Solution().distributeCandies(5, 2))

sai  : 1 | đúng: 3


**Vì sao sai:** ba đứa trẻ **phân biệt** nên mỗi cách gán số kẹo cho từng đứa là một cách riêng: `(1, 2, 2)`, `(2, 1, 2)`, `(2, 2, 1)` là **ba** cách. Đừng gộp bằng `sorted`/`set`.

### Lỗi 4: Bao hàm - loại trừ sai dấu hoặc thiếu hệ số

In [28]:
def sai_dau(n, limit):
    def free(m):
        return comb(m + 2, 2) if m >= 0 else 0
    return free(n) - 3 * free(n - (limit + 1)) - 3 * free(n - 2 * (limit + 1)) - free(n - 3 * (limit + 1))   # sai dấu số hạng thứ 3


def sai_thieu_he_so(n, limit):
    def free(m):
        return comb(m + 2, 2) if m >= 0 else 0
    return free(n) - free(n - (limit + 1))                # thiếu hệ số 3 (chọn đứa nào vi phạm)


for n, limit in [(5, 2), (7, 3), (9, 3)]:
    print(f"n={n}, limit={limit}: sai dấu = {sai_dau(n, limit)}, thiếu hệ số = {sai_thieu_he_so(n, limit)} | đúng = {Solution().distributeCandies(n, limit)}")

n=5, limit=2: sai dấu = 3, thiếu hệ số = 15 | đúng = 3
n=7, limit=3: sai dấu = 6, thiếu hệ số = 26 | đúng = 6
n=9, limit=3: sai dấu = -17, thiếu hệ số = 34 | đúng = 1


**Vì sao sai:** bao hàm - loại trừ có dấu **`+ − + −`** xen kẽ và các hệ số **`C(3, j) = 1, 3, 3, 1`** (có `C(3, j)` cách chọn tập `j` đứa trẻ vi phạm). Sai dấu hoặc thiếu hệ số cho kết quả sai ở những `n` đủ lớn (khi các tập giao khác rỗng). Khi `n` nhỏ (như `n = 5`, `limit = 2`) các số hạng sau bằng `0` nên lỗi dấu chưa lộ ra: hãy thử **nhiều** giá trị.

### Lỗi 5: Dùng `comb` với số âm

In [29]:
try:
    print(comb(-1 + 2, 2))                  # comb(1, 2) = 0: ổn
    print(comb(-3 + 2, 2))                  # comb(-1, 2): ValueError
except ValueError as e:
    print("ValueError:", e)
print("đúng: bọc lại, trả 0 khi m < 0 ->", distribute_inclusion_exclusion(5, 2))

0
ValueError: n must be a non-negative integer
đúng: bọc lại, trả 0 khi m < 0 -> 3


**Vì sao sai:** `math.comb(n, k)` yêu cầu **`n, k >= 0`**. Các số hạng `m = n − j(limit + 1)` có thể âm; khi đó số cách phải là `0`, không phải gọi `comb`. Hàm `free(m)` luôn kiểm tra `m >= 0` trước.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "đếm số cách chia / bộ ba có tổng cố định và có cận trên" → **liệt kê có ràng buộc**; khi `n` nhỏ cứ dùng ba vòng lặp.
- Giảm vòng lặp: **biến cuối bị quyết định** (`c = n - a - b`).
- Với giới hạn lớn: **đếm theo khoảng** `[lo, hi]` (Cách 3) hoặc **bao hàm - loại trừ** (Cách 4).
- Stars and bars: `C(m + k − 1, k − 1)` cách chia `m` vật cho `k` người, không giới hạn.
- Luôn **thử nhiều giá trị** khi kiểm tra công thức, bao gồm các biên `n = 3·limit`, `n > 3·limit`, `limit >= n`.
- Kiểm tra công thức bằng brute force trên miền nhỏ trước khi tin.

## 15. Biến thể của bài

1. **Bài II** (cùng series): `n, limit` tới `10^6`; dùng Cách 3 hoặc Cách 4.
2. Chia cho **`k` đứa trẻ** (thay vì 3): bao hàm - loại trừ tổng quát `Σ (-1)^j C(k, j) C(n − j(limit+1) + k − 1, k − 1)`, hoặc DP.
3. Mỗi đứa phải nhận **ít nhất 1** viên: đặt `x' = x − 1`, chia `n − 3` viên với giới hạn `limit − 1`.
4. Mỗi đứa có **giới hạn riêng** `limit[i]`.
5. Đếm số cách khi **các đứa trẻ giống nhau** (không phân biệt): đếm phân hoạch số.
6. Chia kẹo sao cho **đứa thứ nhất nhận nhiều nhất** (hoặc không ít hơn các đứa kia).

Ta thử biến thể 2 (k đứa trẻ) và 3 (mỗi đứa ít nhất 1):

In [30]:
def distribute_k_children(n, limit, k):
    """Biến thể 2: k đứa trẻ, bao hàm - loại trừ."""
    total = 0
    for j in range(k + 1):
        m = n - j * (limit + 1)
        if m < 0:
            break
        total += (-1) ** j * comb(k, j) * comb(m + k - 1, k - 1)
    return total


def distribute_at_least_one(n, limit):
    """Biến thể 3: mỗi đứa nhận từ 1 đến limit viên (3 đứa)."""
    return Solution().distributeCandies(n - 3, limit - 1) if n >= 3 and limit >= 1 else 0


for n, limit in [(5, 2), (10, 4), (12, 5)]:
    assert distribute_k_children(n, limit, 3) == Solution().distributeCandies(n, limit)
print("2. k = 3 khớp bài gốc; k = 4, n = 5, limit = 2:", distribute_k_children(5, 2, 4), "| DP:", distribute_dp(5, 2, children=4))
assert distribute_k_children(5, 2, 4) == distribute_dp(5, 2, children=4)

alo = sum(1 for a in range(1, 4) for b in range(1, 4) for c in range(1, 4) if a + b + c == 6)
print("3. mỗi đứa 1..3, n = 6:", distribute_at_least_one(6, 3), "| đếm tay:", alo)
assert distribute_at_least_one(6, 3) == alo

2. k = 3 khớp bài gốc; k = 4, n = 5, limit = 2: 16 | DP: 16
3. mỗi đứa 1..3, n = 6: 7 | đếm tay: 7


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ba vòng lặp | duyệt mọi `(a, b, c)` | `O(limit³)` | `O(1)` | hiểu đề; đối chiếu |
| 2. Hai vòng lặp, `c = n - a - b` | bỏ một vòng bằng ràng buộc tổng | `O(limit²)` | `O(1)` | **đủ cho Bài I**, dễ viết nhất |
| 3. Một vòng lặp, đếm cặp theo khoảng | `[max(0, r−limit), min(r, limit)]` | `O(limit)` | `O(1)` | **cách nộp bài**; dùng được cho Bài II |
| 4. Bao hàm - loại trừ | `C(n+2,2) − 3C(·) + 3C(·) − C(·)` | `O(1)` | `O(1)` | công thức đóng, `n` rất lớn |
| 5. DP theo số đứa trẻ | `dp[k][s]` | `O(k·n·limit)` | `O(n)` | số đứa trẻ bất kỳ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `list(range(3))`, `comb(5, 2)`, `Solution().distributeCandies(1, 5)`, `Solution().distributeCandies(4, 1)`, `Solution().distributeCandies(6, 2)`.

**Bài 2 (tính tay).** Với `n = 4`, `limit = 2`: liệt kê mọi bộ `(a, b, c)` với `0 <= a, b, c <= 2`, `a + b + c = 4`. Rồi kiểm tra bằng công thức bao hàm - loại trừ.

**Bài 3 (điền TODO).** Hoàn thành `distribute_todo(n, limit)` theo Cách 2.

In [31]:
def distribute_todo(n, limit):
    count = 0
    # TODO 1: for a in range(limit + 1):
    # TODO 2:     for b in range(limit + 1):
    # TODO 3:         c = n - a - b
    # TODO 4:         nếu 0 <= c <= limit thì count += 1
    return count

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `distribute_range_count(n, limit)` bằng kỹ thuật đếm cặp theo khoảng (Cách 3) mà không nhìn code mẫu. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `distribute_min_each(n, limit, lo)`: mỗi đứa nhận **từ `lo` đến `limit`** viên (đặt `x' = x − lo`). (b) Viết `distribute_first_is_max(n, limit)` đếm các cách mà số kẹo của **đứa thứ nhất lớn hơn hoặc bằng** hai đứa còn lại.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[0, 1, 2]`, `10`, `3`, `0` (4 > 3·1), `1` (`(2,2,2)`). Bài 2: các bộ cần tổng 4 với mỗi số tối đa 2.
- **Mức 2 (vừa):** Bài 2: `(0,2,2)`, `(2,0,2)`, `(2,2,0)`, `(1,1,2)`, `(1,2,1)`, `(2,1,1)` -> 6 cách. Kiểm tra bằng công thức với `F(m) = C(m+2, 2)`: `F(4) − 3F(1) + 3F(−2) − F(−5) = 15 − 9 + 0 − 0 = 6`. Bài 3: bốn dòng đã ghi trong comment. Bài 4: `r = n − a`, `lo = max(0, r − limit)`, `hi = min(r, limit)`.
- **Mức 3 (sâu):** Bài 5(a): đặt `x' = x − lo` thì `x' ∈ [0, limit − lo]` và tổng `n − 3·lo`: đếm các `(a, b)` trong `[0, limit − lo]` với `c = (n − 3·lo) − a − b` nằm trong `[0, limit − lo]` (đừng gọi lại hàm gốc vì nó giả định `n >= 1`). Bài 5(b): duyệt `a`, đếm cặp `(b, c)` với `b, c <= a` và `b + c = n − a`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert list(range(3)) == [0, 1, 2] and comb(5, 2) == 10
assert Solution().distributeCandies(1, 5) == 3
assert Solution().distributeCandies(4, 1) == 0
assert Solution().distributeCandies(6, 2) == 1

# Bài 2
ways4 = [(a, b, c) for a in range(3) for b in range(3) for c in range(3) if a + b + c == 4]
assert sorted(ways4) == sorted([(0, 2, 2), (2, 0, 2), (2, 2, 0), (1, 1, 2), (1, 2, 1), (2, 1, 1)])
assert Solution().distributeCandies(4, 2) == 6 == distribute_inclusion_exclusion(4, 2)

# Bài 3
def distribute_todo_ans(n, limit):
    count = 0
    for a in range(limit + 1):
        for b in range(limit + 1):
            c = n - a - b
            if 0 <= c <= limit:
                count += 1
    return count

for n, limit, expected in examples:
    assert distribute_todo_ans(n, limit) == expected

# Bài 4
def distribute_range_count(n, limit):
    total = 0
    for a in range(min(n, limit) + 1):
        r = n - a
        lo, hi = max(0, r - limit), min(r, limit)
        total += max(0, hi - lo + 1)
    return total

for n, limit, expected in examples:
    assert distribute_range_count(n, limit) == expected

# Bài 5(a)
def distribute_min_each(n, limit, lo):
    m, new_limit = n - 3 * lo, limit - lo
    if m < 0 or new_limit < 0:
        return 0
    return sum(1 for a in range(new_limit + 1) for b in range(new_limit + 1)
               if 0 <= m - a - b <= new_limit)

assert distribute_min_each(6, 3, 1) == 7             # mỗi đứa 1..3, tổng 6
assert distribute_min_each(3, 3, 1) == 1             # (1,1,1)
assert distribute_min_each(5, 2, 0) == Solution().distributeCandies(5, 2)

# Bài 5(b)
def distribute_first_is_max(n, limit):
    count = 0
    for a in range(min(n, limit) + 1):
        for b in range(a + 1):
            c = n - a - b
            if 0 <= c <= a:
                count += 1
    return count

brute = sum(1 for a in range(4) for b in range(4) for c in range(4) if a + b + c == 6 and a >= b and a >= c)
assert distribute_first_is_max(6, 3) == brute
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (một vòng lặp theo đứa thứ nhất, đếm cặp theo khoảng)

```python
class Solution:
    def distributeCandies(self, n: int, limit: int) -> int:
        count = 0
        for a in range(min(n, limit) + 1):
            r = n - a
            count += max(0, min(r, limit) - max(0, r - limit) + 1)
        return count
```

**Checklist**

- [ ] Mỗi đứa nhận từ `0` đến `limit` **gồm cả `limit`** (`range(limit + 1)`)?
- [ ] Kiểm tra **cả hai cận** `0 <= c <= limit` khi suy ra `c`?
- [ ] Đếm **có thứ tự** (các đứa trẻ phân biệt), không gộp `(1, 2, 2)` với `(2, 1, 2)`?
- [ ] Nếu dùng bao hàm - loại trừ: dấu `+ − + −`, hệ số `1, 3, 3, 1`, và `m < 0` cho `0`?
- [ ] Đã thử: `n = 3·limit` (đúng `1` cách), `n > 3·limit` (`0`), `limit >= n`?
- [ ] Đã đối chiếu công thức với brute force trên miền nhỏ?
- [ ] Skeleton đúng `class Solution` / `def distributeCandies(self, n: int, limit: int) -> int`?